# JAVELIN trial models

Adapted from the submitted repository, commit `fce9ade29ef6de731407a7fcf14098f30a272cd3`, for the public-data workflow. Current definitions and figure mappings are documented in `docs/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
_candidates = [Path.cwd(), *Path.cwd().parents]
_project = next((p for p in _candidates if (p / "src/mitoxy/data.py").is_file()), None)
if _project is None:
    raise RuntimeError("Launch this notebook within the Mitoxy repository")
os.environ["MITOXY_PROJECT_ROOT"] = str(_project)


# Figure 6 — public-data treatment-interaction analysis

This notebook reconstructs Figure 6 directly from the public JAVELIN Renal
101 and CheckMate 025 supplementary workbooks downloaded by
`00_Paper_Data_Download.ipynb`. It does not import archived project results and
does not compare any estimate, confidence interval, P value, or FDR with a
previously reported value.

Analysis plan:

- Gene scores are frozen before outcome modeling. Each component gene is
  standardized within cohort; multi-gene scores are the equal-weight mean of
  their component z scores and are then standardized to one cohort SD.
- JAVELIN PFS models use Cox proportional hazards regression. Panel A contains
  descriptive median-split subgroups. Panel B tests prespecified continuous
  score-by-treatment interactions adjusted for age, sex, and PD-L1. Its BH FDR
  family contains nine molecular scores, although only four are plotted.
- Panel C fits exploratory Mitoxy-by-treatment interactions within median
  immune/angiogenic strata, with clinical and molecular-context adjustment.
- CheckMate 025 uses logistic regression for ORR and Cox models for PFS/OS.
  Panel D reports Mitoxy from a prespecified family of eight scores × three
  outcomes; BH correction is performed across all 24 interaction tests.

The compact output CSV names and columns are unchanged, so the existing
Figure 6 plotting notebook can be rerun without modification.


In [ ]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from openpyxl import load_workbook
from pyxlsb import open_workbook as open_xlsb
from statsmodels.duration.hazard_regression import PHReg
from scipy.stats import norm


PROJECT_ROOT = Path(
    os.environ["MITOXY_PROJECT_ROOT"]
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_6_clean"
CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "figure_6_public_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "javelin": source_path(
        "JAVELIN Renal 101", "clinical and molecular tables"
    ),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


In [ ]:
MITOXY_GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3_GENES = ["BAK1", "BAX", "BID"]
MYELOID_GENES = ["MYD88", "TNF", "TLR2", "TLR4"]
SPATIAL4_GENES = ["MAPKAP1", "MLST8", "MTOR", "RICTOR"]
ANGIO_GENES = ["VEGFA", "KDR", "ESM1", "PECAM1", "ANGPTL4", "CD34"]
JAVELIN26_GENES = [
    "CD3G", "CD3E", "CD8B", "THEMIS", "TRAT1", "GRAP2", "CD247",
    "CD2", "CD96", "PRF1", "CD6", "IL7R", "ITK", "GPR18", "EOMES",
    "SIT1", "NLRC3", "CD244", "KLRD1", "SH2D1A", "CCL5", "XCL2",
    "CST7", "GFI1", "KCNA3", "PSTPIP1",
]

JAVELIN_TARGET_GENES = list(dict.fromkeys(
    MITOXY_GENES + MYELOID_GENES + SPATIAL4_GENES +
    ANGIO_GENES + JAVELIN26_GENES + ['CD8A','IFNG','CD274','IL6','CXCL1','CXCL2','CXCL3','CXCL8','PTGS2']
))
CHECKMATE_TARGET_GENES = list(dict.fromkeys(
    MITOXY_GENES + MYELOID_GENES + SPATIAL4_GENES
))


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def canonical_id(value):
    if pd.isna(value):
        return ""
    text = re.sub(r"[^A-Za-z0-9]", "", str(value)).upper()
    if len(text) == 33 and text.startswith("X"):
        text = text[1:]
    return text


def unique_headers(values):
    output, counts = [], {}
    for index, value in enumerate(values):
        base = str(value).strip() if value not in (None, "") else f"column_{index}"
        counts[base] = counts.get(base, 0) + 1
        output.append(base if counts[base] == 1 else f"{base}_{counts[base]}")
    return output


def zscore_series(values):
    values = pd.to_numeric(values, errors="coerce").astype(float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        raise ValueError("Cannot standardize a constant or invalid variable")
    return (values - values.mean()) / standard_deviation


def zscore_frame(frame):
    result = frame.apply(pd.to_numeric, errors="coerce").astype(float)
    standard_deviation = result.std(axis=0, ddof=0).replace(0, np.nan)
    result = result.subtract(result.mean(axis=0), axis=1).divide(
        standard_deviation, axis=1
    )
    if result.isna().any().any():
        bad = result.columns[result.isna().any()].tolist()
        raise ValueError(f"Cannot standardize genes with missing/invalid values: {bad}")
    return result


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(values.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if len(valid) == 0:
        return adjusted
    order = valid[np.argsort(values[valid])]
    ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def resolve_numbered_sheet(sheet_names, number):
    patterns = [
        rf"(?:table|supplementary table|supp table|st|s)\s*0*{number}(?:\D|$)",
        rf"^0*{number}(?:\D|$)",
    ]
    matches = []
    for sheet in sheet_names:
        name = normalized(sheet)
        if any(re.search(pattern, name) for pattern in patterns):
            matches.append(sheet)
    if len(matches) == 1:
        return matches[0]
    contains = [sheet for sheet in sheet_names if str(number) in str(sheet)]
    if len(contains) == 1:
        return contains[0]
    raise RuntimeError(
        f"Could not uniquely resolve Supplementary Table {number}. "
        f"Candidates={matches or contains}; sheets={sheet_names}"
    )


def select_column(columns, alternatives, exclude=(), required=True):
    normalized_columns = {column: normalized(column) for column in columns}
    exclude = [normalized(term) for term in exclude]
    scored = []
    for column, text in normalized_columns.items():
        if any(term in text for term in exclude):
            continue
        best = 0
        for alternative in alternatives:
            terms = [normalized(term) for term in alternative]
            if all(term in text for term in terms):
                best = max(best, 100 + 10 * len(terms) - len(text))
        if best:
            scored.append((best, column))
    if not scored:
        if required:
            raise KeyError(
                f"No column matched {alternatives}. Available: {list(columns)}"
            )
        return None
    return max(scored, key=lambda item: (item[0], str(item[1])))[1]


def parse_event(values, column_name):
    numeric = pd.to_numeric(values, errors="coerce")
    observed = set(numeric.dropna().unique())
    name = normalized(column_name)
    if observed and observed.issubset({0, 1}):
        return 1 - numeric if ("censor" in name or "cnsr" in name) else numeric
    text = values.astype(str).str.strip().str.casefold()
    result = pd.Series(np.nan, index=values.index, dtype=float)
    result.loc[text.str.contains(
        r"death|dead|progress|event|yes|^1$", regex=True, na=False
    )] = 1.0
    result.loc[text.str.contains(
        r"censor|alive|no event|no progression|^0$", regex=True, na=False
    )] = 0.0
    return result


def parse_response(values):
    numeric = pd.to_numeric(values, errors="coerce")
    observed = set(numeric.dropna().unique())
    if observed and observed.issubset({0, 1}):
        return numeric.astype(float)
    text = values.astype(str).str.strip().str.upper()
    compact = text.str.replace(r"[^A-Z0-9]+", "", regex=True)
    result = pd.Series(np.nan, index=values.index, dtype=float)
    result.loc[
        compact.isin({"CR", "PR", "CRPR"})
        | text.str.contains(
            r"COMPLETE RESPONSE|PARTIAL RESPONSE|RESPONDER",
            regex=True, na=False,
        )
    ] = 1.0
    result.loc[
        compact.isin({"SD", "PD"})
        | text.str.contains(
            r"STABLE|PROGRESSIVE|NON.?RESPONDER",
            regex=True, na=False,
        )
    ] = 0.0
    return result


def prepare_design(data, score_column, categorical_covariates=(),
                   numeric_covariates=()):
    required = [
        score_column, "treatment", *categorical_covariates,
        *numeric_covariates,
    ]
    complete = data[required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    design = pd.DataFrame(index=complete.index)
    design["score"] = pd.to_numeric(complete[score_column], errors="coerce")
    design["treatment"] = pd.to_numeric(complete["treatment"], errors="coerce")
    for column in numeric_covariates:
        design[column] = zscore_series(complete[column])
    if categorical_covariates:
        dummies = pd.get_dummies(
            complete[list(categorical_covariates)].astype(str),
            prefix=list(categorical_covariates), drop_first=True, dtype=float,
        )
        design = pd.concat([design, dummies], axis=1)
    design["score_by_treatment"] = design["score"] * design["treatment"]
    return complete, design.astype(float)


def cox_interaction(data, score_column, time_column, event_column,
                    categorical_covariates=(), numeric_covariates=()):
    required = [time_column, event_column]
    analysis = data.dropna(subset=required).copy()
    complete, design = prepare_design(
        analysis, score_column, categorical_covariates, numeric_covariates
    )
    outcomes = analysis.loc[complete.index, [time_column, event_column]].astype(float)
    valid = outcomes[time_column].gt(0) & outcomes[event_column].isin([0, 1])
    outcomes, design = outcomes.loc[valid], design.loc[valid]
    fit = PHReg(
        outcomes[time_column], design,
        status=outcomes[event_column], ties="breslow",
    ).fit(disp=0)
    index = design.columns.get_loc("score_by_treatment")
    coefficient = float(fit.params[index])
    standard_error = float(fit.bse[index])
    return {
        "n": len(outcomes),
        "events": int(outcomes[event_column].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[index]),
    }


def cox_treatment_effect(data, time_column="pfs", event_column="pfs_event"):
    complete = data[[time_column, event_column, "treatment"]].dropna().copy()
    complete = complete.loc[
        complete[time_column].gt(0) & complete[event_column].isin([0, 1])
    ]
    design = complete[["treatment"]].astype(float)
    fit = PHReg(
        complete[time_column].astype(float), design,
        status=complete[event_column].astype(float), ties="breslow",
    ).fit(disp=0)
    coefficient, standard_error = float(fit.params[0]), float(fit.bse[0])
    return {
        "n": len(complete),
        "events": int(complete[event_column].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[0]),
    }


def logistic_interaction(data, score_column, response_column="orr"):
    analysis = data.dropna(subset=[response_column]).copy()
    complete, design = prepare_design(analysis, score_column)
    outcome = analysis.loc[complete.index, response_column].astype(float)
    valid = outcome.isin([0, 1])
    outcome, design = outcome.loc[valid], design.loc[valid]
    design = sm.add_constant(design, has_constant="add")
    fit = sm.Logit(outcome, design).fit(disp=False, maxiter=500)
    coefficient = float(fit.params["score_by_treatment"])
    standard_error = float(fit.bse["score_by_treatment"])
    return {
        "n": len(outcome),
        "events": np.nan,
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues["score_by_treatment"]),
    }


def validate_intervals(table, estimate):
    if not table[estimate].gt(0).all():
        raise ValueError(f"Non-positive estimates in {estimate}")
    if not table["ci_low"].gt(0).all():
        raise ValueError("Non-positive confidence limit")
    if not table["ci_high"].gt(table["ci_low"]).all():
        raise ValueError("Invalid confidence-interval ordering")
    if not (
        table["ci_low"].le(table[estimate])
        & table[estimate].le(table["ci_high"])
    ).all():
        raise ValueError("Estimate falls outside its confidence interval")


## Reconstruct the JAVELIN public clinical and expression tables


In [ ]:
JAVELIN_CLINICAL_CACHE = CACHE_DIR / "javelin_table11_clinical.csv"
JAVELIN_EXPRESSION_CACHE = CACHE_DIR / "javelin_table13_target_expression.csv"


def xlsb_values(cell_row):
    return [cell.v for cell in cell_row]


def reconstruct_javelin_clinical(path):
    table=pd.read_excel(path,sheet_name='S11_Clinical_data',header=1,keep_default_na=False,engine='calamine')
    table['canonical_id']=table['ID'].map(canonical_id)
    if table.canonical_id.duplicated().any():raise ValueError('Duplicate JAVELIN clinical IDs')
    return table



def reconstruct_javelin_expression(path,clinical_ids,targets):
    table=pd.read_excel(path,sheet_name='S13_Gene_expression_TPM',header=1,index_col=0,engine='calamine')
    table.index=table.index.astype(str).str.upper().str.strip()
    table=table.loc[table.index.intersection(targets)].groupby(level=0).mean()
    table.columns=[canonical_id(c) for c in table.columns]
    table=table.loc[:,table.columns.isin(clinical_ids)].T
    if table.index.duplicated().any():raise ValueError('Duplicate JAVELIN expression IDs')
    if len(table)<100:raise ValueError('Insufficient JAVELIN clinical/expression overlap')
    table.index.name='canonical_id'
    return table.reset_index()



if JAVELIN_CLINICAL_CACHE.exists():
    javelin_clinical = pd.read_csv(JAVELIN_CLINICAL_CACHE, dtype=str)
    print("Loading cached JAVELIN clinical table.")
else:
    javelin_clinical = reconstruct_javelin_clinical(RAW["javelin"])
    javelin_clinical.to_csv(JAVELIN_CLINICAL_CACHE, index=False)

clinical_ids = set(javelin_clinical["canonical_id"].dropna().map(canonical_id))
if JAVELIN_EXPRESSION_CACHE.exists():
    javelin_expression = pd.read_csv(JAVELIN_EXPRESSION_CACHE)
    cached_genes = set(javelin_expression.columns) - {"canonical_id"}
    cached_overlap = len(set(javelin_expression["canonical_id"].map(canonical_id)) & clinical_ids)
    if not set(JAVELIN_TARGET_GENES).issubset(cached_genes) or cached_overlap < 100:
        print("Discarding incomplete JAVELIN expression cache.")
        JAVELIN_EXPRESSION_CACHE.unlink()
        javelin_expression = reconstruct_javelin_expression(
            RAW["javelin"], clinical_ids, JAVELIN_TARGET_GENES
        )
        javelin_expression.to_csv(JAVELIN_EXPRESSION_CACHE, index=False)
    else:
        print("Loading cached JAVELIN target-expression table.")
else:
    javelin_expression = reconstruct_javelin_expression(
        RAW["javelin"], clinical_ids, JAVELIN_TARGET_GENES
    )
    javelin_expression.to_csv(JAVELIN_EXPRESSION_CACHE, index=False)

javelin_expression["canonical_id"] = javelin_expression["canonical_id"].map(canonical_id)
javelin_clinical["canonical_id"] = javelin_clinical["canonical_id"].map(canonical_id)
missing = sorted(set(JAVELIN_TARGET_GENES) - set(javelin_expression.columns))
if missing:
    raise KeyError(f"JAVELIN Table 13 is missing target genes: {missing}")

print("JAVELIN clinical identifiers:", len(clinical_ids))
print("JAVELIN expression identifiers:", javelin_expression["canonical_id"].nunique())
print("JAVELIN clinical/expression overlap:", len(
    set(javelin_expression["canonical_id"]) & clinical_ids
))

In [ ]:
# The public JAVELIN clinical table uses compact field names in the released
# workbook. Resolve exact aliases first, then semantic fallbacks.
def exact_or_semantic(columns, exact_aliases, alternatives, exclude=()):
    lookup = {normalized(column): column for column in columns}
    for alias in exact_aliases:
        if normalized(alias) in lookup:
            return lookup[normalized(alias)]
    return select_column(columns, alternatives, exclude=exclude)


j_id = exact_or_semantic(javelin_clinical.columns, ["ID"], [["id"]])
j_pfs = exact_or_semantic(
    javelin_clinical.columns, ["PFS_P"],
    [["pfs", "time"], ["pfs", "month"], ["pfs"]],
    exclude=["cnsr", "censor", "event", "status"],
)
j_pfs_status = exact_or_semantic(
    javelin_clinical.columns, ["PFS_P_CNSR"],
    [["pfs", "cnsr"], ["pfs", "censor"], ["pfs", "event"], ["pfs", "status"]],
)
j_age = exact_or_semantic(javelin_clinical.columns, ["AGE"], [["age"]])
j_sex = exact_or_semantic(javelin_clinical.columns, ["SEX"], [["sex"], ["gender"]])
j_treatment = exact_or_semantic(
    javelin_clinical.columns, ["TRT01P"],
    [["treatment"], ["randomized", "arm"], ["arm"]],
)
j_pdl1 = exact_or_semantic(
    javelin_clinical.columns, ["PDL1FL"],
    [["pdl1"], ["pd", "l1"]],
)
j_cluster = exact_or_semantic(
    javelin_clinical.columns, ["TCGA_cluster"],
    [["tcga", "cluster"], ["molecular", "cluster"], ["cluster"]],
)

javelin_gene_values = javelin_expression.set_index("canonical_id")[JAVELIN_TARGET_GENES]
javelin_gene_values = javelin_gene_values.apply(pd.to_numeric, errors="coerce")
if javelin_gene_values.isna().any().any():
    raise ValueError("JAVELIN target-gene expression contains missing values")
javelin_gene_z = zscore_frame(javelin_gene_values)

javelin_scores = pd.DataFrame(index=javelin_gene_z.index)
javelin_scores["Equal4_mean_gene_z"] = javelin_gene_z[MITOXY_GENES].mean(axis=1)
javelin_scores["Mitoxy"] = zscore_series(javelin_gene_z[MITOXY_GENES].mean(axis=1))
javelin_scores["Core3"] = zscore_series(javelin_gene_z[CORE3_GENES].mean(axis=1))
javelin_scores["Myeloid trigger"] = zscore_series(
    javelin_gene_z[MYELOID_GENES].mean(axis=1)
)
javelin_scores["Spatial4"] = zscore_series(
    javelin_gene_z[SPATIAL4_GENES].mean(axis=1)
)
javelin_scores["ANGIO"] = zscore_series(javelin_gene_z[ANGIO_GENES].mean(axis=1))
javelin_scores["JAVELIN26"] = zscore_series(
    javelin_gene_z[JAVELIN26_GENES].mean(axis=1)
)
for gene in ["SLC7A11", "BID", "BAK1", "BAX", "RICTOR"]:
    javelin_scores[gene] = javelin_gene_z[gene]
javelin_scores['TEFF']=zscore_series(javelin_gene_z[['CD8A','EOMES','PRF1','IFNG','CD274']].mean(axis=1))
javelin_scores['MCDERMOTT_MYELOID']=zscore_series(javelin_gene_z[['IL6','CXCL1','CXCL2','CXCL3','CXCL8','PTGS2']].mean(axis=1))
javelin_scores.index.name = "canonical_id"

javelin = javelin_scores.reset_index().merge(
    javelin_clinical, on="canonical_id", how="inner", validate="one_to_one"
)
javelin["pfs"] = pd.to_numeric(javelin[j_pfs], errors="coerce")
javelin["pfs_event"] = parse_event(javelin[j_pfs_status], j_pfs_status)
javelin["age"] = pd.to_numeric(javelin[j_age], errors="coerce")
javelin["sex"] = javelin[j_sex].astype(str).str.strip()
javelin["pdl1"] = javelin[j_pdl1].astype(str).str.strip()
javelin["tcga_cluster"] = javelin[j_cluster].astype(str).str.strip()
javelin["treatment_label"] = javelin[j_treatment].astype(str).str.strip()
treatment_text = javelin["treatment_label"].str.casefold()
javelin["treatment"] = np.select(
    [
        treatment_text.str.contains(r"avelumab|axitinib|comb", regex=True),
        treatment_text.str.contains(r"sunitinib", regex=True),
    ],
    [1.0, 0.0], default=np.nan,
)

javelin.to_csv(OUTPUT_DIR/'JAVELIN_all_patients.csv',index=False)

javelin_model = javelin.dropna(subset=[
    "pfs", "pfs_event", "age", "sex", "pdl1", "tcga_cluster", "treatment"
]).copy()
if not set(javelin_model["treatment"].unique()).issubset({0.0, 1.0}):
    raise ValueError("JAVELIN treatment coding failed")
if javelin_model["treatment"].nunique() != 2:
    raise ValueError("Both JAVELIN treatment arms were not recovered")

print("JAVELIN matched model rows:", len(javelin_model))
print("JAVELIN PFS events:", int(javelin_model["pfs_event"].sum()))
print("Treatment counts:")
display(javelin_model["treatment_label"].value_counts().to_frame("n"))
display(pd.DataFrame({
    "resolved_field": ["ID", "PFS", "PFS status", "age", "sex", "treatment", "PD-L1", "TCGA cluster"],
    "source_column": [j_id, j_pfs, j_pfs_status, j_age, j_sex, j_treatment, j_pdl1, j_cluster],
}))


## Panel A — descriptive two-dimensional JAVELIN subgroups


In [ ]:
mitoxy_median = javelin_model["Mitoxy"].median()
immuno_median = javelin_model["JAVELIN26"].median()
javelin_model["mitoxy_group"] = np.where(
    javelin_model["Mitoxy"].ge(mitoxy_median), "high", "low"
)
javelin_model["immuno_group"] = np.where(
    javelin_model["JAVELIN26"].ge(immuno_median), "high", "low"
)

panel_a_rows = []
for mitoxy_group, immuno_group in [
    ("high", "high"), ("high", "low"),
    ("low", "high"), ("low", "low"),
]:
    subset = javelin_model.loc[
        javelin_model["mitoxy_group"].eq(mitoxy_group)
        & javelin_model["immuno_group"].eq(immuno_group)
    ]
    result = cox_treatment_effect(subset)
    panel_a_rows.append({
        "subgroup": (
            f"Mitoxy-{mitoxy_group} / JAVELIN26-{immuno_group}"
        ),
        "n": result["n"], "events": result["events"],
        "hr": result["effect"], "ci_low": result["ci_low"],
        "ci_high": result["ci_high"], "p_value": result["p_value"],
    })

panel_a = pd.DataFrame(panel_a_rows)
validate_intervals(panel_a, "hr")
panel_a.to_csv(OUTPUT_DIR / "panel_a_2d_subgroups.csv", index=False)
display(panel_a)


## Panel B — formal continuous JAVELIN interactions


In [ ]:
# The multiplicity family is defined independently of which rows are shown in
# the figure. This prevents the displayed subset from changing its own FDR.
javelin_fdr_family = [
    "Mitoxy", "Core3", "Myeloid trigger", "Spatial4",
    "SLC7A11", "BID", "BAK1", "BAX", "RICTOR",
]

family_rows = []
for score in javelin_fdr_family:
    result = cox_interaction(
        javelin_model, score, "pfs", "pfs_event",
        categorical_covariates=["sex", "pdl1"],
        numeric_covariates=["age"],
    )
    family_rows.append({
        "score": score, "n": result["n"], "events": result["events"],
        "hr": result["effect"], "ci_low": result["ci_low"],
        "ci_high": result["ci_high"], "p_value": result["p_value"],
    })

javelin_family_results = pd.DataFrame(family_rows)
javelin_family_results["fdr"] = bh_adjust(javelin_family_results["p_value"])
javelin_family_results.to_csv(
    OUTPUT_DIR / "panel_b_full_fdr_family.csv", index=False
)

display_order = ["Mitoxy", "Core3", "Myeloid trigger", "SLC7A11"]
panel_b = (
    javelin_family_results.set_index("score").loc[display_order]
    .reset_index()
)
validate_intervals(panel_b, "hr")
panel_b.to_csv(OUTPUT_DIR / "panel_b_formal_interactions.csv", index=False)

print("Full nine-score BH family:")
display(javelin_family_results)
print("Panel B plotting subset:")
display(panel_b)


## Panel C — exploratory immune/angiogenic context strata


In [ ]:
panel_c_rows = []


def stable_context_interaction(data):
    """Fit the prespecified adjusted interaction with a stable Cox start.

    The ridge solution is used only as an optimizer starting point. The
    returned coefficient, confidence interval, and P value all come from the
    final unpenalized partial-likelihood model.
    """
    analysis = data.dropna(subset=["pfs", "pfs_event"]).copy()
    complete, design = prepare_design(
        analysis, "Mitoxy",
        categorical_covariates=["sex", "pdl1", "tcga_cluster"],
        numeric_covariates=["age", "ANGIO", "JAVELIN26"],
    )
    outcomes = analysis.loc[complete.index, ["pfs", "pfs_event"]].astype(float)
    valid = outcomes["pfs"].gt(0) & outcomes["pfs_event"].isin([0, 1])
    outcomes, design = outcomes.loc[valid], design.loc[valid]

    # Retain the three focal terms, then add nuisance columns only when they
    # increase the design rank. This removes empty/sparse dummy columns that
    # can otherwise make the Hessian singular within a median-defined stratum.
    focal = ["score", "treatment", "score_by_treatment"]
    retained = []
    for column in focal + [c for c in design.columns if c not in focal]:
        candidate = retained + [column]
        matrix = design[candidate].to_numpy(dtype=float)
        if np.linalg.matrix_rank(matrix) == len(candidate):
            retained.append(column)
    design = design[retained]
    if "score_by_treatment" not in design.columns:
        raise RuntimeError("The focal interaction is not identifiable")

    model = PHReg(
        outcomes["pfs"], design,
        status=outcomes["pfs_event"], ties="breslow",
    )
    fit = None
    diagnostics = []
    for alpha in (1e-4, 1e-3, 1e-2):
        try:
            ridge = model.fit_regularized(
                alpha=alpha, L1_wt=0.0, maxiter=2000
            )
            candidate = model.fit(
                start_params=np.asarray(ridge.params, dtype=float),
                method="bfgs", maxiter=2000, disp=0,
            )
            interaction_index = design.columns.get_loc("score_by_treatment")
            values = np.array([
                candidate.params[interaction_index],
                candidate.bse[interaction_index],
                candidate.pvalues[interaction_index],
            ], dtype=float)
            if np.isfinite(values).all():
                fit = candidate
                break
            diagnostics.append(f"alpha={alpha}: non-finite inference")
        except Exception as error:
            diagnostics.append(f"alpha={alpha}: {type(error).__name__}: {error}")
    if fit is None:
        raise RuntimeError(
            "Adjusted stratified Cox model did not converge after stable "
            "initialization. " + " | ".join(diagnostics)
        )

    index = design.columns.get_loc("score_by_treatment")
    coefficient = float(fit.params[index])
    standard_error = float(fit.bse[index])
    return {
        "n": len(outcomes),
        "events": int(outcomes["pfs_event"].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[index]),
        "n_parameters": len(retained),
    }


for context, display_context in [
    ("JAVELIN26", "JAVELIN26"), ("ANGIO", "ANGIO")
]:
    median = javelin_model[context].median()
    for stratum, mask in [
        ("Low", javelin_model[context].lt(median)),
        ("High", javelin_model[context].ge(median)),
    ]:
        subset = javelin_model.loc[mask].copy()
        result = stable_context_interaction(subset)
        panel_c_rows.append({
            "context_stratum": f"{display_context}-{stratum.lower()}",
            "context": display_context, "stratum": stratum,
            "n": result["n"], "events": result["events"],
            "hr": result["effect"], "ci_low": result["ci_low"],
            "ci_high": result["ci_high"], "p_value": result["p_value"],
            "n_parameters": result["n_parameters"],
        })

panel_c = pd.DataFrame(panel_c_rows)
validate_intervals(panel_c, "hr")
panel_c.to_csv(OUTPUT_DIR / "panel_c_context_interactions.csv", index=False)
display(panel_c)


## Reconstruct the CheckMate 025 clinical and expression tables


In [ ]:
javelin_model.to_csv(OUTPUT_DIR / 'JAVELIN_analysis_patients.csv', index=False)
